# nuReasoning challenge submission

This tutorial builds the official test-set JSON: **one entry per clip**, each holding a 5 s ego-frame trajectory and that clip's multiple-choice answers.

Two ways to fill the file:

1. **Path A — one model.** Train nuVLA with `--reasoning_mode vqa --vqa_root` so the VLM text target is VQA. The same `final/` checkpoint plans and answers.
2. **Path B — two models.** A planning-only nuVLA checkpoint samples trajectories **in this process**; a separately trained reasoning VLM (vLLM) answers the questions.

The hosted evaluator is CPU-only and scores the JSON you upload; it does not run your model. Full write-up: [`docs/submission.md`](../docs/submission.md).

Prerequisites: the devkit (`pip install -e .`) and the test split under `dataset/data/test` (or `$NUREASONING_TEST_ROOT`). The first code cell `chdir`s to the repository root so paths match the README. Training cells are opt-in.

nuVLA train flags use underscores (`--data_root`, `--vqa_root`). Challenge submission flags use hyphens (`--data-root`, `--checkpoint-dir`).

In [ ]:
import json
import os
from pathlib import Path

_here = Path.cwd()
REPO_ROOT = _here if (_here / "nureasoning").is_dir() else _here.parent
os.chdir(REPO_ROOT)

TEST_ROOT = Path(os.environ.get("NUREASONING_TEST_ROOT", "./dataset/data/test"))
TRAIN_ROOT = Path(os.environ.get("NUREASONING_TRAIN_ROOT", "./dataset/data/train"))
VALIDATION_ROOT = Path(os.environ.get("NUREASONING_VALIDATION_ROOT", "./dataset/data/validation"))
VQA_ROOT = Path(os.environ.get("NUREASONING_VQA_ROOT", "./vqa_output_train"))
VLA_QA_WORKSPACE = Path(os.environ.get("NUREASONING_VLA_QA_WORKSPACE", "./nureasoning_vla_qa_workspace"))
VLA_QA_CHECKPOINT = Path(os.environ.get("NUREASONING_VLA_QA_CHECKPOINT", str(VLA_QA_WORKSPACE / "final")))
VLA_CHECKPOINT = Path(os.environ.get("NUREASONING_VLA_CHECKPOINT", "./nureasoning_vla_workspace/final"))
OUTPUT = Path(os.environ.get("NUREASONING_CHALLENGE_OUTPUT", "./challenge_submission.json"))
MAX_CLIPS = int(os.environ.get("NUREASONING_MAX_CLIPS", "2"))
VLM_MODEL = os.environ.get("NUREASONING_VLM_MODEL", "Qwen/Qwen3-VL-2B-Instruct")

print("REPO_ROOT", REPO_ROOT)
print("TEST_ROOT", TEST_ROOT, "exists", TEST_ROOT.is_dir())
print("VLA_QA_CHECKPOINT", VLA_QA_CHECKPOINT, "exists", VLA_QA_CHECKPOINT.is_dir())
print("VLA_CHECKPOINT", VLA_CHECKPOINT, "exists", VLA_CHECKPOINT.is_dir())
print("MAX_CLIPS", MAX_CLIPS, "(0 = all 1000 test clips)")

## 1. Inspect a test clip

Each directory is a 10 s clip through key frame 100: eight cameras, ego state (no future trajectory), and `reasoning_questions.json` with no gold answers.

In [ ]:
from nureasoning.common.clips import discover_clips
from nureasoning.submission.clips import load_clip_sample

clips = discover_clips(str(TEST_ROOT), max_clips=1)
assert clips, f"No clips under {TEST_ROOT}"
sample = load_clip_sample(clips[0])
print("clip", sample["clip"])
print("key_frame_index", sample["key_frame_index"])
print("cameras", sorted((sample.get("image_paths") or {}).keys()))
print("n_questions", sample["num_questions"])
for q in sample["questions"]:
    print(" -", q["question_id"], q["category"], q["question_type"], list((q.get("choices") or {}).keys()))

## 2. Dry-run the submission file

Constant-velocity planning + stub answers. Use this to check the JSON shape before loading a checkpoint. `MAX_CLIPS` defaults to 2 in this notebook; set `NUREASONING_MAX_CLIPS=0` for the full 1,000 clips.

In [ ]:
!python -m nureasoning.submission.challenge \
    --data-root "{TEST_ROOT}" \
    --provider split \
    --planning-provider constant_velocity \
    --reasoning-provider stub \
    --max-clips {MAX_CLIPS} \
    --output "{OUTPUT}"

payload = json.loads(OUTPUT.read_text())
print("meta", payload["meta"])
clip0 = payload["clips"][0]
print("clip", clip0["clip"], "target_frame", clip0["target_frame_index"])
print("trajectory shape", len(clip0["trajectory"]), "x", len(clip0["trajectory"][0]))
print("n_answers", len(clip0["answers"]))
if clip0["answers"]:
    print("first answer", clip0["answers"][0])

## 3. Path A — nuVLA trained with VQA (one model)

Generate VQA from the training annotations, then train nuVLA with `--reasoning_mode vqa --vqa_root`. The VLM text target is those multiple-choice questions (same prompt/answer format as the challenge). The action expert still trains on every sample. After training, use the `final/` directory under the workspace.

Set `NUREASONING_RUN_VQA=1` and `NUREASONING_RUN_TRAINING=1` to actually run those steps (hours on a GPU cluster).

In [ ]:
RUN_TRAINING = os.environ.get("NUREASONING_RUN_TRAINING", "0") == "1"
RUN_VQA = os.environ.get("NUREASONING_RUN_VQA", "0") == "1"

if RUN_VQA:
    !python -m nureasoning.vqa.generate \
        --data-root "{TRAIN_ROOT}" \
        --output "{VQA_ROOT}"
else:
    print("Skipping VQA generation. Set NUREASONING_RUN_VQA=1 to run it.")

if RUN_TRAINING:
    !python -m nureasoning.nuvla.train \
        --data_root "{TRAIN_ROOT}" \
        --test_data_root "{VALIDATION_ROOT}" \
        --vlm_model_path "{VLM_MODEL}" \
        --reasoning_mode vqa \
        --vqa_root "{VQA_ROOT}" \
        --output_dir "{VLA_QA_WORKSPACE}"
else:
    print("Skipping nuVLA+QA training. Set NUREASONING_RUN_TRAINING=1 to run it.")

Generate the official file from that `final/` checkpoint. `--provider nuvla` loads the VLM adapter and action expert once and, per clip, samples a trajectory then answers every question with the same VLM. Planning uses `--planning-prompt` (default `auto`). Challenge answers always use the VQA question wording.

In [ ]:
RUN_NUVLA_SUBMISSION = os.environ.get("NUREASONING_RUN_NUVLA_SUBMISSION", "0") == "1"
if RUN_NUVLA_SUBMISSION and VLA_QA_CHECKPOINT.is_dir():
    !python -m nureasoning.submission.challenge \
        --data-root "{TEST_ROOT}" \
        --provider nuvla \
        --checkpoint-dir "{VLA_QA_CHECKPOINT}" \
        --max-clips {MAX_CLIPS} \
        --output "{OUTPUT}"
else:
    print(
        "Skipping nuVLA+QA submission. Set NUREASONING_RUN_NUVLA_SUBMISSION=1 "
        f"and point NUREASONING_VLA_QA_CHECKPOINT at a checkpoint (current: {VLA_QA_CHECKPOINT})."
    )

## 4. Path B — split models (nuVLA planning + reasoning SFT)

Use a **planning-only** nuVLA checkpoint (train without `--vqa_root`; use `…/final`) for trajectories, and a separately fine-tuned reasoning VLM (`nureasoning.reasoning.train`) for answers.

The next cell is **one process** that does both:

1. **Planning inference** — `--planning-provider vla` loads `VLA_CHECKPOINT` here and samples a 5 s ego-frame trajectory at each clip's key frame. There is no planner server.
2. **Reasoning** — `--reasoning-provider api` sends questions to a vLLM server. Start it in another terminal (vLLM env; see `docs/installation.md`):

```bash
VLLM_USE_FLASHINFER_SAMPLER=0 vllm serve ./reasoning_workspace/merged_qwen3.5-4b-multiframe \
  --served-model-name nureasoning-4b-sft \
  --max-model-len 16384 \
  --gpu-memory-utilization 0.7 \
  --dtype bfloat16 \
  --mm-processor-kwargs '{"max_pixels": 200704}' \
  --enable-prefix-caching
```

`--max-model-len` only caps sequence length. `--gpu-memory-utilization` is what leaves room on the same GPU for nuVLA planning. Prefix caching speeds up questions that share the same camera frames; it does not shrink the KV pool.

Without a server, set `NUREASONING_REASONING_PROVIDER=vlm` to load a Hugging Face / nuVLA VLM in-process instead. `--planning-provider constant_velocity` is the no-checkpoint sanity baseline used in section 2.

In [ ]:
RUN_SPLIT_SUBMISSION = os.environ.get("NUREASONING_RUN_SPLIT_SUBMISSION", "0") == "1"
REASONING_PROVIDER = os.environ.get("NUREASONING_REASONING_PROVIDER", "api")
API_MODEL = os.environ.get("VLLM_MODEL", "nureasoning-4b-sft")

if RUN_SPLIT_SUBMISSION and Path(VLA_CHECKPOINT).is_dir():
    !python -m nureasoning.submission.challenge \
        --data-root "{TEST_ROOT}" \
        --provider split \
        --planning-provider vla --checkpoint-dir "{VLA_CHECKPOINT}" \
        --reasoning-provider {REASONING_PROVIDER} --api-model {API_MODEL} \
        --max-clips {MAX_CLIPS} \
        --output "{OUTPUT}"
else:
    print(
        "Skipping split submission. Set NUREASONING_RUN_SPLIT_SUBMISSION=1, "
        "provide NUREASONING_VLA_CHECKPOINT (planning inference), "
        "and start vLLM unless NUREASONING_REASONING_PROVIDER=vlm."
    )

## 5. Validate the JSON

Checks clip ids, 51×3 finite ego trajectories, unique `question_id`s, and (when `--data-root` is the full test tree) that every clip and question is present. Skip coverage checks if you generated with `--max-clips`.

In [ ]:
from nureasoning.submission.format import load_submission, validate_submission

if OUTPUT.is_file():
    payload = load_submission(str(OUTPUT))
    # Coverage against the full test tree only when this file has every clip.
    check_root = None if MAX_CLIPS > 0 else str(TEST_ROOT)
    errors = validate_submission(payload, data_root=check_root)
    if errors:
        print("Validation issues:")
        for err in errors:
            print(" -", err)
    else:
        print(
            f"OK: {OUTPUT}  clips={payload['meta'].get('num_clips')}  "
            f"answers={payload['meta'].get('num_answers')}"
        )
else:
    print(f"No submission at {OUTPUT}; run a generation cell first.")

## 6. Scoring

Upload `challenge_submission.json` to the Hugging Face competition:
[nureasoning-challenge-2026](https://huggingface.co/spaces/nureasoning/nureasoning-challenge-2026).
The CPU evaluator computes:

- **Planning score** — mean target-frame-gated nuReasoning planning score (NPS) over every expected clip. Missing or invalid trajectories score zero.
- **Reasoning score** — exact multiple-choice accuracy over every expected question. Missing, duplicate, or invalid answers score zero.
- **Final score** = 0.75 × planning + 0.25 × reasoning

Generation is strict by default: if any clip's trajectory or any question fails, no file is written. `--allow-partial` is for debugging only.

Check the file locally before upload:

```bash
python -m nureasoning.submission.challenge \
  --data-root ./dataset/data/test \
  --validate-only ./challenge_submission.json
```